In [7]:
import pickle
from hubo_qaoa.utils.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian
from hubo_qaoa.utils.gfa_utils import gfa_file_to_graph

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QAOAAnsatz

from qiskit_aer import AerSimulator
from qiskit_aer.backends.backendconfiguration import AerBackendConfiguration
from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy

In [18]:
filename = 'trivial'
copy_numbers = [1.0,1.0,1.0]
times='01'

In [19]:
with open(f'/lustre/scratch127/qpg/jc59/new_hubo_formulation/compilation.{filename}.couplinggrid.extra1.times{times}.four0.0.six1.0.pkl', 'rb') as f:
    res = pickle.load(f)

In [20]:
res['best_rzz']

{'count': 184,
 'depth': 140,
 'layers': 8,
 'layout': Layout({
 8: <Qubit register=(9, "q"), index=0>,
 7: <Qubit register=(9, "q"), index=1>,
 6: <Qubit register=(9, "q"), index=2>,
 5: <Qubit register=(9, "q"), index=3>,
 4: <Qubit register=(9, "q"), index=4>,
 3: <Qubit register=(9, "q"), index=5>,
 2: <Qubit register=(9, "q"), index=6>,
 1: <Qubit register=(9, "q"), index=7>,
 0: <Qubit register=(9, "q"), index=8>
 }),
 'circuit': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x7ff55e9eab90>}

In [21]:
res['best_rzz']['circuit'].draw(fold=-1)

global phase: 4.5077
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  ┌───┐                  ┌───┐                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 
q_0 -> 0 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────■────────────────────────────────────────────────────────────────■────────────────■────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────■───────────────■──────────────────────────────────────────────────────────────────────────■────────────────────────────────────────────■───────────────────────────────────────■──┤ X ├──■───────────────┤ X ├───────────────X─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────■──────────────────────────────────────■─────────────────────────────────────────────────────■───────────────────────────■─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────■────────────■─────────────────────────────────────────────────────────────────────────────■──────────────────────────■───X──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                                                                                                                                                                                      ┌───┐   ┌───────────┐ │ZZ(-0.625) ┌───┐                          ┌───┐    ┌──────────┐ │ZZ(-1.25) ┌───┐ │                                                                                                                                                                                                                                │             ┌─┴─┐                                                                      ┌─┴─┐┌───┐                                   ┌─┴─┐                                   ┌─┴─┐└─┬─┘  │               └─┬─┘    ┌───┐      │                                                                                                                          ┌───┐┌───────────┐ │ZZ(-0.625)            ┌───┐           │                                                   ┌─┴─┐                       ┌─┴─┐                                                                                    

In [ ]:
filepath = f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/{filename}.gfa'
graph, n, V, total_weight = gfa_file_to_graph(filepath, copy_numbers)
T = total_weight

full_hamiltonian = graph_to_hubo_hamiltonian(graph, n, total_weight, lamda=10, constraint_terms=1.0)
qaoa_cost_op = QAOAAnsatz(
    full_hamiltonian,
    mixer_operator=QuantumCircuit(n * T),
    initial_state=QuantumCircuit(n * T)
)
extended_swap_strat = ExtendedSwapStrategy.from_grid(n, T)


Keeping constraints at times: [0 1]


In [23]:
basis_gates=["sx", "x", "rz", "rx", "rzz", "cz", "id", "cx", "swap"]

backend_options = dict(
    method='statevector',
    device='GPU',
    precision='single',
    basis_gates=basis_gates
)


config = AerSimulator._DEFAULT_CONFIGURATION
config["n_qubits"] = n * T
config["basis_gates"] = basis_gates
config = AerBackendConfiguration.from_dict(config)
backend = AerSimulator(configuration=config, coupling_map=extended_swap_strat._coupling_map, **backend_options)

In [24]:
backend_tqaoa = transpile(qaoa_cost_op, optimization_level=3, backend=backend, basis_gates=basis_gates)
backend_tqaoa.draw(fold=-1)

/nfs/users/nfs_j/jc59/quantumwork/pangenome/.venv/lib/python3.10/site-packages/qiskit/compiler/transpiler.py:269: UserWarning: Providing `coupling_map` and/or `basis_gates` along with `backend` is not recommended, as this will invalidate the backend's gate durations and error rates.
  pm = generate_preset_pass_manager(


global phase: (-20.625)*γ[0]
         ┌───────────────────┐                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      
q_6 -> 0 ┤ Rz((-0.625)*γ[0]) ├────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────X───────────────────────────────────────────────────────────────────────────────────────────────────

In [28]:
print(backend_tqaoa.count_ops())
print(res['best_rzz']['circuit'].count_ops())

print(backend_tqaoa.depth(lambda instr: len(instr.qubits) > 1))
print(res['best_rzz']['circuit'].depth(lambda instr: len(instr.qubits) > 1))

OrderedDict([('cx', 186), ('rz', 66), ('swap', 55), ('rzz', 23)])
OrderedDict([('cx', 111), ('rzz', 63), ('rz', 26), ('swap', 10)])
240
140
